# Chess DQN on Google Colab — Universal Game Engine

Universal Game Engine のバックエンド（Bun + gRPC）を **Colab 内で起動**し、
Python (PyTorch) の DQN エージェント（Double DQN + 合法手マスク）が gRPC の `Reset` / `Step` でチェスを自己対戦しながら学習します。
Python 側はチェスのルールを持たず、合法手・終局判定はすべてサーバーの `ChessRuleset` が担当します。学習済みモデルは Google Drive に保存され、`uge_rl.serve` で実際の対局相手として使えます。

**観測 / 行動**（`ChessTensorAdapter`）: 観測は自分視点の盤面 64 マス（黒は上下反転）+ キャスリング権 4 + アンパッサン 1 + 50 手カウンタ 1 + 同形回数 1 = 71 要素で、NN 入力では駒種ごとの one-hot などの 19 チャンネル × 8 × 8 に展開します。行動は「移動先マス × 28 種（方向 8 / ナイト 8 / 昇格 12）」= 1792 通りで、合法手以外の Q 値は `-inf` でマスクします。

**注意**: DQN は 1 手ごとの TD 学習なので、チェスのように 1 局が長く報酬が終局時にしか出ないゲームでは AlphaZero 版（`chess_alphazero_colab.ipynb`）より学習が難しいです。まずは手数上限（`--max-moves`）で引き分け打ち切りにして、対ランダム勝率が上がるかを見てください。

**手順**: ランタイム → 「ランタイムのタイプを変更」で GPU (T4) を選んでから、上から順に実行してください。

| ステップ | 内容 |
| --- | --- |
| 1 | Google Drive をマウント（モデル保存先） |
| 2 | リポジトリを clone、Bun をインストール |
| 3 | バックエンドを `RL_MODE=true` でバックグラウンド起動 |
| 4 | 学習 (`uge_rl.train`) |
| 5 | 評価 (`uge_rl.evaluate`) と学習曲線 |


## 1. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/UniversalGameEngine/models'
import os; os.makedirs(MODEL_DIR, exist_ok=True)
print('models will be saved to', MODEL_DIR)

## 2. リポジトリの取得と Bun のインストール

private リポジトリの場合は `REPO_URL` を `https://<GITHUB_TOKEN>@github.com/...` の形式にしてください。

In [ ]:
import os
REPO_URL = 'https://github.com/takumi-mr/UniversalGameEngine.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

if not os.path.exists('/content/UniversalGameEngine'):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/UniversalGameEngine
%cd /content/UniversalGameEngine

# Bun
!curl -fsSL https://bun.sh/install | bash > /dev/null 2>&1
os.environ['PATH'] = '/root/.bun/bin:' + os.environ['PATH']
!bun --version

# 依存関係（postinstall の git hook 設定は Colab では不要なのでスキップ）
!bun install --frozen-lockfile --ignore-scripts

## 3. バックエンドをバックグラウンド起動

`RL_MODE=true` にすると Redis / MongoDB なしのインメモリ動作になります。ログは `server.log` に出ます。

In [ ]:
import subprocess, sys, time
sys.path.insert(0, '/content/UniversalGameEngine/apps/ml')

env = dict(os.environ, RL_MODE='true', PORT='3000', GRPC_PORT='50051')
server = subprocess.Popen(
    ['bun', 'run', 'apps/backend/server.ts'],
    cwd='/content/UniversalGameEngine',
    env=env,
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)

from uge_rl.env import wait_for_server
wait_for_server('localhost:50051', timeout_sec=90)
print('gRPC server ready (pid', server.pid, ')')
!tail -n 5 /content/server.log

## 4. Python 依存関係

In [ ]:
!pip install -q -r apps/ml/requirements.txt
import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

## 5. 学習

`--episodes` を増やすほど強くなります。チェスは 1 エピソードが最大 `--max-moves` 手（既定 200）なので、オセロより 1 エピソードあたり数倍時間がかかります。
`--eps-decay-steps` は総ステップ数（≒ エピソード数 × 平均手数）の 1/3 程度を目安にしてください。
途中経過のチェックポイントも `--save-every` ごとに同じパスへ上書き保存され、中断した場合は `--resume {MODEL_PATH}` で再開できます。

In [ ]:
EPISODES = 3000  #@param {type:"integer"}
MAX_MOVES = 200  #@param {type:"integer"}
MODEL_PATH = f'{MODEL_DIR}/chess_dqn.pt'

!cd apps/ml && python -m uge_rl.train     --game chess     --address localhost:50051     --episodes {EPISODES}     --max-moves {MAX_MOVES}     --eps-decay-steps 200000     --buffer-size 200000     --train-every 2     --out {MODEL_PATH}     --eval-every 100 --eval-games 10     --save-every 100 --log-every 10

## 6. 評価（ランダムプレイヤーとの対戦）

`--max-moves` を超えた対局は引き分けになります（省略時はチェックポイントの設定を使います）。

In [ ]:
!cd apps/ml && python -m uge_rl.evaluate --checkpoint {MODEL_PATH} --address localhost:50051 --games 50

## 7. 学習曲線（対ランダム勝率）

In [ ]:
import json
import matplotlib.pyplot as plt

meta = json.load(open(MODEL_PATH.replace('.pt', '.json')))
hist = meta.get('eval_history', [])
if hist:
    ep, wr = zip(*hist)
    plt.plot(ep, wr, marker='o')
    plt.axhline(0.5, ls='--', c='gray')
    plt.xlabel('episode'); plt.ylabel('win rate vs random'); plt.ylim(0, 1)
    plt.title(f"Chess DQN ({meta['total_steps']} steps)")
    plt.show()
print({k: meta[k] for k in ('game_type', 'arch', 'total_steps', 'train_steps', 'saved_at', 'git_commit')})

## 8. 保存されたファイルと、モデルと対局する方法

- `chess_dqn.pt` — PyTorch の state_dict + メタ情報（`uge_rl.checkpoint.load_checkpoint` で復元。`format: uge-rl/dqn/v1`）
- `chess_dqn.json` — メタ情報のみ（ゲーム種別・観測形状・行動数・学習ステップ数・学習曲線）

ローカルで対局するには、`.pt` を `models/` にダウンロードしてから

```bash
task rl                                      # バックエンド（RL_MODE=true）
cd apps/frontend && bun dev                  # フロントエンド
cd apps/ml && python -m uge_rl.serve --checkpoint ../../models/chess_dqn.pt   # モデルのボットサーバー（DQN は Q 値の argmax で指す）
```

フロントエンドでチェスの「カスタムマッチ」→ 相手を **☁️ gRPC External** にして部屋を作ると、`serve` がその席を見つけて指し始めます。

In [ ]:
!ls -la {MODEL_DIR}

## 9. 後片付け（任意）

In [ ]:
server.terminate()
print('server stopped')